## Step 1
Merge volume bids with generator info  
Input: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/  
Output: s3://thesis--ec331-s3/enriched-volume-bids/

In [5]:
import awswrangler as wr
import pandas as pd

# Define paths to the folders containing Parquet files
folder1_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000.parquet/"
# folder2_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/"

# First, let's verify the folder(s) exist and see what's inside
active_folders = []

try:
    files1 = wr.s3.list_objects(folder1_path)
    print(f"Found {len(files1)} files in folder1")
    active_folders.append(folder1_path)
except Exception as e:
    print(f"Error listing objects in folder1: {e}")

# Only try folder2 if it's defined
if 'folder2_path' in locals():
    try:
        files2 = wr.s3.list_objects(folder2_path)
        print(f"Found {len(files2)} files in folder2")
        active_folders.append(folder2_path)
    except Exception as e:
        print(f"Error listing objects in folder2: {e}")

# Read and combine the dataframes from active folders
df = None

# First approach: read each folder separately and combine if needed
if len(active_folders) > 0:
    dataframes = []
    
    try:
        df1 = wr.s3.read_parquet(path=folder1_path)
        print(f"Successfully read folder1: {folder1_path}")
        print(f"Shape: {df1.shape}")
        dataframes.append(df1)
    except Exception as e:
        print(f"Error reading folder1: {e}")
    
    # Only try folder2 if it's defined
    if 'folder2_path' in locals():
        try:
            df2 = wr.s3.read_parquet(path=folder2_path)
            print(f"Successfully read folder2: {folder2_path}")
            print(f"Shape: {df2.shape}")
            dataframes.append(df2)
        except Exception as e:
            print(f"Error reading folder2: {e}")
    
    # Combine dataframes if any were read successfully
    if len(dataframes) > 0:
        if len(dataframes) == 1:
            df = dataframes[0]
            print(f"Using dataframe from single folder, shape: {df.shape}")
        else:
            df = pd.concat(dataframes, ignore_index=True)
            print(f"Combined dataframe from {len(dataframes)} folders, shape: {df.shape}")
        
        print(df.head())
    
    # Alternative approach as fallback: read all folders at once
    if df is None and len(active_folders) > 0:
        try:
            df = wr.s3.read_parquet(path=active_folders, dataset=True)
            print(f"Successfully read all folders using dataset=True approach")
            print(f"Combined shape: {df.shape}")
            print(df.head())
        except Exception as e:
            print(f"Error reading all folders at once: {e}")
else:
    print("No valid folders found to process")

Found 1090 files in folder1
Successfully read folder1: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000.parquet/
Shape: (21227616, 28)
Using dataframe from single folder, shape: (21227616, 28)
   I  BIDS  BIDOFFERPERIOD    1      DUID    BIDTYPE          TRADINGDATE  \
0  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC  2023/10/16 00:00:00   
1  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC  2023/10/16 00:00:00   
2  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC  2023/10/16 00:00:00   
3  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC  2023/10/16 00:00:00   
4  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC  2023/10/16 00:00:00   

         OFFERDATETIME  PERIODID  MAXAVAIL  ...  BANDAVAIL2  BANDAVAIL3  \
0  2023/10/09 12:20:25       1.0      27.0  ...         2.0         2.0   
1  2023/10/09 12:20:25       2.0      27.0  ...         2.0         2.0   
2  2023/10/09 12:20:25       3.0      27.0  ...         2.0         2.0 

In [4]:
df.columns

Index(['I', 'BIDS', 'BIDOFFERPERIOD', '1', 'DUID', 'BIDTYPE', 'TRADINGDATE',
       'OFFERDATETIME', 'PERIODID', 'MAXAVAIL', 'FIXEDLOAD', 'RAMPUPRATE',
       'RAMPDOWNRATE', 'ENABLEMENTMIN', 'ENABLEMENTMAX', 'LOWBREAKPOINT',
       'HIGHBREAKPOINT', 'BANDAVAIL1', 'BANDAVAIL2', 'BANDAVAIL3',
       'BANDAVAIL4', 'BANDAVAIL5', 'BANDAVAIL6', 'BANDAVAIL7', 'BANDAVAIL8',
       'BANDAVAIL9', 'BANDAVAIL10', 'PASAAVAILABILITY'],
      dtype='object')

In [4]:
# To find the first (earliest) datetime
first_datetime = df['TRADINGDATE'].min()

# To find the last (latest) datetime
last_datetime = df['TRADINGDATE'].max()

# Print the results
print(f"First datetime: {first_datetime}")
print(f"Last datetime: {last_datetime}")

First datetime: 2023/10/08 00:00:00
Last datetime: 2023/10/31 00:00:00


In [6]:
import awswrangler as wr
import pandas as pd
import time
import os
import gc
from datetime import datetime

# Path to generator info file
generator_info_path = "s3://thesis--ec331-s3/AEMO-Participants - Sheet1.csv"

# Get smaller dataframe first
try:
    generator_info_df = wr.s3.read_csv(path=generator_info_path)
    print(f"Successfully read generator info: {generator_info_path}")
    print(f"Generator info shape: {generator_info_df.shape}")
    print("Generator info columns:", generator_info_df.columns.tolist())
except Exception as e:
    print(f"Error reading generator info: {e}")
    generator_info_df = None

# Define chunk processing function
def process_chunk(chunk_df, generator_info_df, chunk_num, total_chunks, timestamp):
    try:
        print(f"Processing chunk {chunk_num}/{total_chunks} with {len(chunk_df)} rows")
        
        # Merge this chunk
        if 'DUID' in chunk_df.columns and generator_info_df is not None and 'DUID' in generator_info_df.columns:
            enriched_chunk = pd.merge(
                chunk_df,
                generator_info_df,
                on='DUID',
                how='left'
            )
            
            # Save to S3
            chunk_output_filename = f"enriched_volume_bids_{timestamp}_chunk{chunk_num}of{total_chunks}.parquet"
            chunk_output_path = f"s3://thesis--ec331-s3/enriched-volume-bids/{chunk_output_filename}"
            
            # Save this chunk
            chunk_start_time = time.time()
            wr.s3.to_parquet(
                df=enriched_chunk,
                path=chunk_output_path,
                index=False,
                compression="snappy"
            )
            
            chunk_elapsed_time = time.time() - chunk_start_time
            print(f"✓ Chunk {chunk_num}/{total_chunks} saved to {chunk_output_path}")
            print(f"  Chunk save completed in {chunk_elapsed_time:.2f} seconds")
            print(f"  Chunk size: {len(enriched_chunk)} rows, {enriched_chunk.shape[1]} columns")
            
            # Clear memory
            del enriched_chunk
            
        else:
            print("Skipping chunk - missing required columns")
            
    except Exception as e:
        print(f"Error processing chunk {chunk_num}: {str(e)}")
    
    # Force garbage collection
    gc.collect()

# Define ultra-small chunk size to avoid memory issues
chunk_size = 10000  # Try with a very small chunk size

# Calculate total size from df
total_rows = len(df)
total_chunks = (total_rows + chunk_size - 1) // chunk_size

# Create timestamp for filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Processing {total_rows} rows in {total_chunks} chunks of {chunk_size} rows each")

# Process in extremely small chunks with immediate cleanup
for i in range(total_chunks):
    # Get chunk indices
    start_idx = i * chunk_size
    end_idx = min(start_idx + chunk_size, total_rows)
    
    # Extract chunk (copy to avoid reference to original df)
    chunk = df.iloc[start_idx:end_idx].copy()
    
    # Process chunk
    process_chunk(chunk, generator_info_df, i+1, total_chunks, timestamp)
    
    # Clear chunk immediately
    del chunk
    gc.collect()
    
    # Print memory stats if psutil is available (optional)
    try:
        import psutil
        process = psutil.Process(os.getpid())
        print(f"Memory usage after chunk {i+1}: {process.memory_info().rss / 1e9:.2f} GB")
    except:
        pass

print("All chunks processed successfully")

Successfully read generator info: s3://thesis--ec331-s3/AEMO-Participants - Sheet1.csv
Generator info shape: (532, 20)
Generator info columns: ['Participant', 'Station Name', 'Region', 'Dispatch Type', 'Category', 'Classification', 'Fuel Source - Primary', 'Fuel Source - Descriptor', 'Technology Type - Primary', 'Technology Type - Descriptor', 'Units', 'Aggregation', 'DUID', 'Reg Cap generation (MW)', 'Max Cap generation (MW)', 'Max ROC/Min generation', 'Reg Cap consumption (MW)', 'Max Cap consumption (MW)', 'Max ROC/Min consumption', 'Comments']
Processing 21227616 rows in 2123 chunks of 10000 rows each
Processing chunk 1/2123 with 10000 rows
✓ Chunk 1/2123 saved to s3://thesis--ec331-s3/enriched-volume-bids/enriched_volume_bids_20250312_102609_chunk1of2123.parquet
  Chunk save completed in 0.36 seconds
  Chunk size: 10000 rows, 47 columns
Memory usage after chunk 1: 12.84 GB
Processing chunk 2/2123 with 10000 rows
✓ Chunk 2/2123 saved to s3://thesis--ec331-s3/enriched-volume-bids/enr

In [18]:
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)

# Display the first 5 rows again
print(enriched_df.tail())

          I  BIDS  BIDOFFERPERIOD    1      DUID    BIDTYPE  \
21021115  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021116  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021117  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021118  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021119  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   

                  TRADINGDATE        OFFERDATETIME  PERIODID  MAXAVAIL  \
21021115  2023/11/15 00:00:00  2023/11/16 03:51:42     284.0      27.0   
21021116  2023/11/15 00:00:00  2023/11/16 03:51:42     285.0      27.0   
21021117  2023/11/15 00:00:00  2023/11/16 03:51:42     286.0      27.0   
21021118  2023/11/15 00:00:00  2023/11/16 03:51:42     287.0      27.0   
21021119  2023/11/15 00:00:00  2023/11/16 03:51:42     288.0      27.0   

          FIXEDLOAD  RAMPUPRATE  RAMPDOWNRATE  ENABLEMENTMIN  ENABLEMENTMAX  \
21021115        NaN         NaN           NaN            0.0           50.0   
21021116        Na

In [20]:
# Filtering rows where 'Region Dispatch' is NOT NaN
filtered_df = enriched_df[enriched_df['Region'].notna()]

# Display the last 5 rows of the filtered dataframe
print(filtered_df.tail())

          I  BIDS  BIDOFFERPERIOD    1     DUID    BIDTYPE  \
18440923  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440924  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440925  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440926  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440927  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   

                  TRADINGDATE        OFFERDATETIME  PERIODID  MAXAVAIL  \
18440923  2023/11/15 00:00:00  2023/11/16 03:48:32     285.0       5.0   
18440924  2023/11/15 00:00:00  2023/11/16 03:48:32     286.0       5.0   
18440925  2023/11/15 00:00:00  2023/11/16 03:48:32     287.0       5.0   
18440926  2023/11/15 00:00:00  2023/11/16 03:48:32     288.0       5.0   
18440927  2023/11/15 00:00:00  2023/11/16 02:08:31     288.0       5.0   

          FIXEDLOAD  RAMPUPRATE  RAMPDOWNRATE  ENABLEMENTMIN  ENABLEMENTMAX  \
18440923        NaN         NaN           NaN            0.0           10.0   
18440924        NaN     

In [21]:
print(enriched_df.isna().sum())

I                                      0
BIDS                                   0
BIDOFFERPERIOD                         0
1                                      0
DUID                                   0
BIDTYPE                                0
TRADINGDATE                            0
OFFERDATETIME                          0
PERIODID                               0
MAXAVAIL                               0
FIXEDLOAD                       21021120
RAMPUPRATE                      21021120
RAMPDOWNRATE                    21021120
ENABLEMENTMIN                          0
ENABLEMENTMAX                          0
LOWBREAKPOINT                          0
HIGHBREAKPOINT                         0
BANDAVAIL1                             0
BANDAVAIL2                             0
BANDAVAIL3                             0
BANDAVAIL4                             0
BANDAVAIL5                             0
BANDAVAIL6                             0
BANDAVAIL7                             0
BANDAVAIL8      